# LLM-powered Financial Fraud Detection System

# Dataset Summary — Credit Card Fraud Detection

# About the Dataset
This dataset contains **284,807 real credit card transactions** made by 
European cardholders over **2 days in September 2013**. It was collected 
and shared by the Machine Learning Group at Université Libre de Bruxelles 
(ULB) for fraud detection research.

# Goal
Build a machine learning model that can automatically detect fraudulent 
transactions in real-time — flagging suspicious activity before the bank 
loses money.

# Class Distribution
- **284,315 transactions** are legitimate (Class = 0) — 99.827%
- **492 transactions** are fraudulent (Class = 1) — 0.173%
- This is a **severely imbalanced dataset** — fraud is extremely rare

# Column Descriptions

| Column | Type | Description |
|--------|------|-------------|
| Time | Numerical | Seconds elapsed since the first transaction in the dataset |
| V1–V28 | Numerical | **PCA-transformed features** — original features were anonymized by the bank for privacy and legal compliance. Cannot be reverse-engineered. |
| Amount | Numerical | Transaction amount in Euros — this is a real, untransformed value |
| Class | Binary | **Target variable** — 0 = Legitimate, 1 = Fraud |

# Why V1–V28 Cannot Be Named
The bank applied **PCA (Principal Component Analysis)** to the original 
28 features before releasing the dataset. This was done for two reasons:

1. **Privacy law** — GDPR prohibits sharing identifiable customer data publicly
2. **Security** — exposing real feature names would help fraudsters 
   reverse-engineer the detection system

Despite anonymization, the mathematical fraud patterns are fully preserved. 
The model learns from these patterns just as effectively as it would 
from named features.

# Key Challenges
1. **Class imbalance** — 99.83% legitimate vs 0.17% fraud makes accuracy 
   a useless metric. We use F1 Score instead.
2. **No feature names** — V1–V28 are anonymous. SHAP explainability helps 
   us understand which features matter most.
3. **Real-world complexity** — Fraud patterns are subtle and overlap 
   with legitimate transactions.

# What This Dataset Represents in Production
In real banks, data scientists never see raw customer data. They always 
work with anonymized or encrypted features — exactly like V1–V28 here. 
This dataset accurately mirrors real production fraud detection workflows 
at financial institutions like JP Morgan, Goldman Sachs, and Capital One.

# Solution Approach
1. **XGBoost classifier** —  a strong candidate model for imbalanced tabular data
2. **SMOTE or class-weighting** — handle class imbalance on training data only
3. **SHAP explainability** — explain WHY each transaction was flagged
4. **LangChain RAG pipeline** — retrieve relevant FATF/FinCEN regulations 
   for each fraud flag
5. **FastAPI + Streamlit** — deploy as a production web application

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/raw/creditcard.csv")
print(f"Dataset Loaded Successfully")
print(f"Shape: {df.shape}")
df.head()
df_raw = df.copy()



Dataset Loaded Successfully
Shape: (284807, 31)


In [51]:
print(f"Total Columns: {df.shape[1]}")
print(f"ALL THE COLUMNS ARE ")
print(df.columns.tolist())
print(df.columns)


Total Columns: 31
ALL THE COLUMNS ARE 
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


In [52]:
print(df[['Time', 'Amount', 'Class']].describe())

                Time         Amount          Class
count  284807.000000  284807.000000  284807.000000
mean    94813.859575      88.349619       0.001727
std     47488.145955     250.120109       0.041527
min         0.000000       0.000000       0.000000
25%     54201.500000       5.600000       0.000000
50%     84692.000000      22.000000       0.000000
75%    139320.500000      77.165000       0.000000
max    172792.000000   25691.160000       1.000000


In [53]:

print(df['Class'].count())
print(f"\nFraud: {df['Class'].sum()} Fraud Transactions")
print(f"Legitimate: {(df['Class']==0).sum()} Legitimate Transactions")
print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")

print("Unique target values:", df['Class'].unique())
print("Number of target classes:", df['Class'].nunique())

284807

Fraud: 492 Fraud Transactions
Legitimate: 284315 Legitimate Transactions
Fraud rate: 0.173%
Unique target values: [0 1]
Number of target classes: 2


In [54]:
print(df.duplicated().sum())

missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

if missing_values.sum() == 0:
    print("No missing values found in the dataset.")
else:
    print(missing_values[missing_values > 0])

# Analyze non-numeric columns
    
non_numeric_cols = df.select_dtypes(exclude=['number']).columns.tolist()

print("Non-numeric columns:", non_numeric_cols)
print("Number of non-numeric columns:", len(non_numeric_cols))


1081
Total missing values: 0
No missing values found in the dataset.
Non-numeric columns: []
Number of non-numeric columns: 0


In [55]:
from IPython.display import display

print("Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

print("\nMissing Values:")
print(df.isnull().sum().to_string())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nClass Distribution:")
print(df['Class'].value_counts())

print("\nClass Distribution Percentage:")
print(df['Class'].value_counts(normalize=True) * 100)

Dataset Shape:
(284807, 31)

First 5 Rows:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000



Missing Values:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0

Total Missing Values:
0

Duplicate Rows:
1081

Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Class Distribution Percentage:
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


# Data Cleaning Part


In [56]:
# =========================
# Data Cleaning
# =========================

print("Original dataset shape:", df.shape)

# 1. Check and remove duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows before cleaning:", duplicate_count)

print("\nDuplicate rows by class:")
print(df[df.duplicated()]['Class'].value_counts())

df = df.drop_duplicates().reset_index(drop=True)

print("\nDuplicate rows after cleaning:", df.duplicated().sum())
print("Dataset shape after removing duplicates:", df.shape)

print("\nClass distribution after duplicate removal:")
print(df['Class'].value_counts())

print("\nClass distribution percentage after duplicate removal:")
print(df['Class'].value_counts(normalize=True) * 100)


# 2. Check missing values
missing_values = df.isnull().sum()

print("\nTotal missing values:", missing_values.sum())

if missing_values.sum() == 0:
    print("No missing values found in the dataset.")
else:
    print(missing_values[missing_values > 0])


# 3. Validate target column
print("\nUnique target values:", df['Class'].unique())
print("Number of target classes:", df['Class'].nunique())

df['Class'] = df['Class'].astype(int)


# 4. Check invalid Amount and Time values
print("\nNegative Amount values:", (df['Amount'] < 0).sum())
print("Negative Time values:", (df['Time'] < 0).sum())


# 5. Check non-numeric columns
non_numeric_cols = df.select_dtypes(exclude=['number']).columns.tolist()

print("\nNon-numeric columns:", non_numeric_cols)
print("Number of non-numeric columns:", len(non_numeric_cols))


# 6. Zero amount transaction analysis
df['IsZeroAmount'] = (df['Amount'] == 0).astype(int)

zero_amount = df[df['Amount'] == 0]

print("\nZero amount transactions:", len(zero_amount))

print("\nZero amount class distribution:")
print(zero_amount['Class'].value_counts())

print("\nZero amount fraud rate:")
print(zero_amount['Class'].mean() * 100)


# 7. Feature engineering
df['LogAmount'] = np.log1p(df['Amount'])

df['Hour'] = (df['Time'] / 3600).astype(int)
df['HourOfDay'] = df['Hour'] % 24

df['HourSin'] = np.sin(2 * np.pi * df['HourOfDay'] / 24)
df['HourCos'] = np.cos(2 * np.pi * df['HourOfDay'] / 24)


# 8. Final validation after feature engineering
numeric_df = df.select_dtypes(include=['number'])

print("\nInfinite values in dataset:", np.isinf(numeric_df).sum().sum())
print("NaN values after feature engineering:", numeric_df.isnull().sum().sum())

constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

print("\nConstant columns:", constant_cols)
print("Number of constant columns:", len(constant_cols))

print("\nFinal cleaned dataset shape:", df.shape)

print("\nNew engineered columns:")
print(['LogAmount', 'Hour', 'HourOfDay', 'HourSin', 'HourCos', 'IsZeroAmount'])

df.head()

Original dataset shape: (284807, 31)
Duplicate rows before cleaning: 1081

Duplicate rows by class:
Class
0    1062
1      19
Name: count, dtype: int64

Duplicate rows after cleaning: 0
Dataset shape after removing duplicates: (283726, 31)

Class distribution after duplicate removal:
Class
0    283253
1       473
Name: count, dtype: int64

Class distribution percentage after duplicate removal:
Class
0    99.83329
1     0.16671
Name: proportion, dtype: float64

Total missing values: 0
No missing values found in the dataset.

Unique target values: [0 1]
Number of target classes: 2

Negative Amount values: 0
Negative Time values: 0

Non-numeric columns: []
Number of non-numeric columns: 0

Zero amount transactions: 1808

Zero amount class distribution:
Class
0    1783
1      25
Name: count, dtype: int64

Zero amount fraud rate:
1.3827433628318584

Infinite values in dataset: 0
NaN values after feature engineering: 0

Constant columns: []
Number of constant columns: 0

Final cleaned datase

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V27,V28,Amount,Class,IsZeroAmount,LogAmount,Hour,HourOfDay,HourSin,HourCos
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.133558,-0.021053,149.62,0,0,5.014760,0,0,0.0,1.0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.008983,0.014724,2.69,0,0,1.305626,0,0,0.0,1.0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.055353,-0.059752,378.66,0,0,5.939276,0,0,0.0,1.0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.062723,0.061458,123.50,0,0,4.824306,0,0,0.0,1.0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.219422,0.215153,69.99,0,0,4.262539,0,0,0.0,1.0


In [57]:
print("Final dataset shape:", df.shape)
print("New engineered columns:")
print(['LogAmount', 'Hour', 'HourOfDay', 'HourSin', 'HourCos', 'IsZeroAmount'])

Final dataset shape: (283726, 37)
New engineered columns:
['LogAmount', 'Hour', 'HourOfDay', 'HourSin', 'HourCos', 'IsZeroAmount']


# FINAL SUMMARY

## 1. Dataset Loading

The credit card fraud detection dataset was loaded successfully.

The original dataset contained:

- **284,807 rows**
- **31 columns**

The main columns in the dataset are:

- `Time`: Seconds elapsed since the first transaction in the dataset
- `V1` to `V28`: PCA-transformed numerical features
- `Amount`: Transaction amount
- `Class`: Target variable  
  - `0` = Legitimate transaction  
  - `1` = Fraudulent transaction

The features `V1` to `V28` are anonymized because the original banking features were transformed using PCA for privacy and security reasons.

---

## 2. Basic Data Inspection

Basic data inspection was performed to understand the structure and quality of the dataset.

The following checks were performed:

- Dataset shape using `df.shape`
- First few rows using `df.head()`
- Column names using `df.columns`
- Data types and non-null counts using `df.info()`
- Statistical summary using `df.describe()`
- Missing values check
- Duplicate rows check
- Class distribution check

The dataset was found to be highly imbalanced. Legitimate transactions are the majority class, while fraudulent transactions are the minority class.

This class imbalance is important because accuracy alone is not a reliable evaluation metric for fraud detection.

---

## 3. Missing Values

Missing values were checked using:

    df.isnull().sum()

The dataset contained:

- **0 missing values**

This means no missing value imputation was required.

Since every column had complete values, the dataset was suitable to continue with cleaning and feature engineering.

---

## 4. Duplicate Rows

Duplicate rows were checked using:

    df.duplicated().sum()

The dataset contained:

- **1,081 duplicate rows**

Duplicate rows were removed using:

    df = df.drop_duplicates().reset_index(drop=True)

Duplicate records were removed to avoid biased analysis and to prevent the model from learning repeated transaction patterns.

After removing duplicates, the dataset was reset using `reset_index(drop=True)` so that the row index remained clean and continuous.

---

## 5. Target Variable Validation

The target column `Class` was checked to make sure it contains only valid binary values:

- `0` = Legitimate transaction
- `1` = Fraudulent transaction

The target column was converted to integer type using:

    df['Class'] = df['Class'].astype(int)

This ensures that the target variable is in the correct format for machine learning model training.

---

## 6. Invalid Value Checks

The `Amount` and `Time` columns were checked for invalid negative values.

The following checks were performed:

    (df['Amount'] < 0).sum()
    (df['Time'] < 0).sum()

Transaction amount and transaction time should not be negative.

No invalid negative values were found, so no rows needed to be removed for this reason.

---

## 7. Zero Amount Transactions

Zero-amount transactions were checked because transactions with `Amount = 0` may represent unusual or suspicious behavior.

The dataset contained zero-amount transactions from both classes:

- Legitimate transactions with zero amount
- Fraudulent transactions with zero amount

These transactions were **not removed** because some fraud cases also had zero transaction amount. Removing them could remove useful fraud patterns from the dataset.

A new binary feature called `IsZeroAmount` was created:

    df['IsZeroAmount'] = (df['Amount'] == 0).astype(int)

This feature helps the model identify whether a transaction has zero amount and allows the model to learn whether zero-amount transactions are related to fraud risk.

---

## 8. Outlier Handling

Outliers were not removed blindly.

In fraud detection, unusual transactions may actually be fraudulent transactions. Therefore, removing outliers could remove important fraud patterns from the dataset.

Instead of deleting suspicious or extreme transactions, a safer approach was used:

- Keep suspicious or extreme transactions
- Use `LogAmount` to reduce the effect of very large transaction values
- Use scaling later during preprocessing
- Handle class imbalance during model training

This approach is safer because fraud detection models need to learn from unusual transaction behavior instead of removing it.

---

## 9. Feature Engineering

Feature engineering was performed to create more useful features from the existing dataset columns.

No external data was added. The new features were created only from existing columns such as `Amount` and `Time`.

Before feature engineering, the dataset had:

- **31 original columns**

After feature engineering, new columns were added.

The new engineered columns are:

- `LogAmount`
- `Hour`
- `HourOfDay`
- `HourSin`
- `HourCos`
- `IsZeroAmount`

If all 6 engineered features are added, the dataset increases from **31 columns** to **37 columns**.

---

## 10. LogAmount Feature

The original `Amount` column is highly skewed because most transactions are small, while a few transactions have very large amounts.

To reduce this skewness, a new column called `LogAmount` was created:

    df['LogAmount'] = np.log1p(df['Amount'])

The function `np.log1p()` applies:

    log(Amount + 1)

This is useful because it handles zero values safely and reduces the impact of very large transaction amounts.

Example:

- Amount = 10 → LogAmount ≈ 2.40
- Amount = 100 → LogAmount ≈ 4.61
- Amount = 1000 → LogAmount ≈ 6.91
- Amount = 10000 → LogAmount ≈ 9.21

`LogAmount` helps make the transaction amount distribution smoother and easier for the model to learn from.

---

## 11. Hour Feature

The original `Time` column represents seconds since the first transaction.

A new column called `Hour` was created by converting seconds into hours:

    df['Hour'] = (df['Time'] / 3600).astype(int)

This makes the time feature easier to understand and useful for time-based analysis.

For example:

- 3600 seconds = 1 hour
- 7200 seconds = 2 hours
- 86400 seconds = 24 hours

The `Hour` feature is useful for analyzing fraud behavior over time.

---

## 12. HourOfDay Feature

The `Hour` column keeps increasing over time, but daily transaction behavior repeats every 24 hours.

A new column called `HourOfDay` was created:

    df['HourOfDay'] = df['Hour'] % 24

This converts time into a 24-hour daily cycle with values from 0 to 23.

For example:

- Hour 0 → HourOfDay 0
- Hour 24 → HourOfDay 0
- Hour 25 → HourOfDay 1
- Hour 47 → HourOfDay 23

This feature helps identify daily transaction patterns and can be useful for fraud rate analysis by hour of day.

---

## 13. HourSin and HourCos Features

Time is cyclical. For example, hour `23` and hour `0` are close in real life, but numerically they look far apart.

To handle this cyclic nature of time, two new features were created:

    df['HourSin'] = np.sin(2 * np.pi * df['HourOfDay'] / 24)
    df['HourCos'] = np.cos(2 * np.pi * df['HourOfDay'] / 24)

These features represent time as a cycle instead of a straight line.

This helps the model understand that:

- Hour 23 and hour 0 are close
- Daily transaction behavior repeats every 24 hours
- Time-based fraud patterns may follow a daily cycle

`HourSin` and `HourCos` are better for model training than using only raw hour values.

---

## 14. Final Feature Engineering Conclusion

Feature engineering improved the dataset by creating better representations of `Amount` and `Time`.

For EDA, all engineered features are useful:

- `LogAmount`
- `Hour`
- `HourOfDay`
- `HourSin`
- `HourCos`
- `IsZeroAmount`

For model training, the most useful engineered features are:

- `LogAmount`
- `HourSin`
- `HourCos`
- `IsZeroAmount`

The original columns `Amount`, `Time`, `Hour`, and `HourOfDay` can later be dropped before model training to avoid repeated information.

Recommended model feature selection:

    X = df.drop(['Class', 'Time', 'Amount', 'Hour', 'HourOfDay'], axis=1)
    y = df['Class']

This keeps the useful PCA features `V1` to `V28` and the engineered features that are more suitable for machine learning.

---

## 15. Final Cleaning and Feature Engineering Notes

The dataset did not require heavy cleaning because:

- There were no missing values
- All features were numeric
- The target column contained valid binary values
- Duplicate rows were removed
- Invalid negative values were checked
- Zero-amount transactions were kept because they may contain fraud patterns
- Outliers were not removed blindly because unusual transactions may be important for fraud detection

The cleaned and feature-engineered dataset is now ready for final EDA visualizations and later model training.